In [11]:
import ssl
import certifi
import seaborn as sns
import pandas as pd
import urllib.request
import io
from pathlib import Path

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"

context = ssl.create_default_context(cafile=certifi.where())

with urllib.request.urlopen(url, context=context) as response:
    titanic = pd.read_csv(io.BytesIO(response.read()))

print(titanic.head())



   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


In [12]:
titanic_propre = titanic.copy()

print("Dimensions du dataset :", titanic_propre.shape)

display(titanic_propre.head())

print("Valeurs manquantes par colonne :")
display(titanic_propre.isna().sum())

Dimensions du dataset : (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Valeurs manquantes par colonne :


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [13]:
nombre_doublons = titanic_propre.duplicated().sum()

print("Nombre de lignes dupliquées :", nombre_doublons)

display(
    titanic_propre[titanic_propre.duplicated(keep=False)].head(10)
)

Nombre de lignes dupliquées : 107


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
26,0,3,male,NaN,0,0,7.2250,C,Third,man,True,NaN,Cherbourg,no,True
28,1,3,female,NaN,0,0,7.8792,Q,Third,woman,False,NaN,Queenstown,yes,True
29,0,3,male,NaN,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True
32,1,3,female,NaN,0,0,7.7500,Q,Third,woman,False,NaN,Queenstown,yes,True
37,0,3,male,21.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
42,0,3,male,NaN,0,0,7.8958,C,Third,man,True,NaN,Cherbourg,no,True
45,0,3,male,NaN,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
46,0,3,male,NaN,1,0,15.5000,Q,Third,man,True,NaN,Queenstown,no,False
47,1,3,female,NaN,0,0,7.7500,Q,Third,woman,False,NaN,Queenstown,yes,True


In [14]:
mediane_age_par_classe_sexe = titanic_propre.groupby(
    ["pclass", "sex"]
)["age"].transform("median")

titanic_propre["age"] = titanic_propre["age"].fillna(
    mediane_age_par_classe_sexe
)

In [15]:
ville_plus_frequente = titanic_propre["embark_town"].mode()[0]

titanic_propre["embark_town"] = (
    titanic_propre["embark_town"]
    .astype("object")
    .fillna(ville_plus_frequente)
)

titanic_propre["deck"] = (
    titanic_propre["deck"]
    .astype("object")
    .fillna("Unknown")
)

manquants_apres_nettoyage = titanic_propre[
    ["age", "embark_town", "deck"]
].isna().sum()

display(manquants_apres_nettoyage)

age            0
embark_town    0
deck           0
dtype: int64

In [16]:
colonnes_redondantes = [
    "class",
    "alive",
    "embarked",
    "who",
    "adult_male",
    "alone"
]

titanic_propre = titanic_propre.drop(
    columns=colonnes_redondantes
)

print("Colonnes conservées :")
print(titanic_propre.columns.tolist())

Colonnes conservées :
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'deck', 'embark_town']


In [17]:
titanic_propre["survived"] = titanic_propre["survived"].astype(int)
titanic_propre["pclass"] = titanic_propre["pclass"].astype(int)
titanic_propre["sibsp"] = titanic_propre["sibsp"].astype(int)
titanic_propre["parch"] = titanic_propre["parch"].astype(int)

titanic_propre["age"] = pd.to_numeric(titanic_propre["age"])
titanic_propre["fare"] = pd.to_numeric(titanic_propre["fare"])

titanic_propre["sex"] = titanic_propre["sex"].astype("category")
titanic_propre["embark_town"] = titanic_propre["embark_town"].astype("category")
titanic_propre["deck"] = titanic_propre["deck"].astype("category")

titanic_propre.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    category
 3   age          891 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   deck         891 non-null    category
 8   embark_town  891 non-null    category
dtypes: category(3), float64(2), int64(4)
memory usage: 44.6 KB


In [ ]:
q1 = titanic_propre["fare"].quantile(0.25)
q3 = titanic_propre["fare"].quantile(0.75)
ecart_interquartile = q3 - q1
borne_inferieure = q1 - 1.5 * ecart_interquartile
borne_superieure = q3 + 1.5 * ecart_interquartile
billets_hors_bornes = titanic_propre[
    (titanic_propre["fare"] < borne_inferieure)
    | (titanic_propre["fare"] > borne_superieure)
]
print("Bornes :", round(borne_inferieure, 2), "à", round(borne_superieure, 2))
print("Billets hors bornes :", len(billets_hors_bornes))
print("Tarif maximal :", titanic_propre["fare"].max())


In [18]:
print("Dimensions finales :", titanic_propre.shape)

print("Valeurs manquantes restantes :")
display(titanic_propre.isna().sum())

print("Types des colonnes :")
display(titanic_propre.dtypes)

Dimensions finales : (891, 9)
Valeurs manquantes restantes :


survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
deck           0
embark_town    0
dtype: int64

Types des colonnes :


survived          int64
pclass            int64
sex            category
age             float64
sibsp             int64
parch             int64
fare            float64
deck           category
embark_town    category
dtype: object

In [ ]:
from pathlib import Path

dossier_exercice = Path("exercice-08") if Path("exercice-08").is_dir() else Path(".")
dossier_data = dossier_exercice / "data"
dossier_data.mkdir(parents=True, exist_ok=True)

chemin_fichier = dossier_data / "titanic_propre.csv"

titanic_propre.to_csv(chemin_fichier, index=False)

print("Fichier enregistré :", chemin_fichier.resolve())

### Journal de nettoyage

- **Âge :** 177 valeurs manquantes sont remplacées par la médiane du groupe classe-sexe, pour conserver les passagers tout en tenant compte de leur groupe.
- **Port d’embarquement :** 2 valeurs manquantes sont remplacées par le port le plus fréquent, Southampton. Les conserver permet de garder les 891 lignes ; cela explique pourquoi l’effectif anglais peut être supérieur de 2 au repère du support, qui les retire.
- **Pont :** 688 valeurs manquantes (environ 77 %) sont remplacées par `Unknown`, car aucune imputation fiable n’est possible avec les données disponibles.
- **Doublons :** 107 lignes sont signalées par `duplicated()`. Elles sont conservées : des lignes identiques ne prouvent pas qu’il s’agit du même passager, faute d’identifiant unique.
- **Colonnes redondantes :** 6 colonnes (`class`, `alive`, `embarked`, `who`, `adult_male`, `alone`) sont retirées parce qu’elles répètent ou dérivent d’autres informations.
- **Types :** 4 colonnes sont gardées en entiers, 2 en nombres décimaux et 3 en catégories pour faciliter les analyses.
- **Tarifs atypiques :** la règle des 1,5 écarts interquartiles repère 116 lignes hors bornes. Elles sont conservées, car un tarif extrême peut être réel.
- **Export :** le jeu nettoyé est enregistré en CSV sans index supplémentaire.